# 05 Error Analysis
**Purpose of this notebook:** examine the texts the classifier assigns to the wrong class in E2, in order to describe *what kind* of text defeats shallow stylometric features. The predictions are regenerated with the same seed and the same grouped cross-validation as in E2, so the errors analysed here are exactly the errors reported there.

**Input:** `features.csv` (from 03), `all_texts.csv` (from 02)
**Output:** `results/e5_error_examples.csv`, plus the error table used on the poster

## Step 1: Setup and data

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict

os.makedirs("results", exist_ok=True)
SEED = 42
pd.set_option("display.width", 200)

In [ ]:
from google.colab import files
uploaded = files.upload()      # features.csv 和 all_texts.csv

In [ ]:
FEATURES = ["ttr", "avg_sent_len",
            "noun_ratio", "verb_ratio", "adj_ratio", "adv_ratio", "pron_ratio",
            "sent_intensity",
            "comma_rate", "excl_rate", "quest_rate"]

feat = pd.read_csv("features.csv")
texts = pd.read_csv("all_texts.csv")[["text_id", "text"]]
df = feat.merge(texts, on="text_id", validate="one_to_one")
print(len(df), "texts")

## Step 2: Regenerate the out-of-fold predictions
Class probabilities are kept as well, so that errors can be ordered by how confidently they were made. A confidently wrong prediction is more informative than a borderline one.

In [ ]:
def make_model():
    return Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
X, y, g = df[FEATURES], df["is_llm"], df["tconst"]

proba = cross_val_predict(make_model(), X, y, groups=g, cv=cv, method="predict_proba")[:, 1]
df["p_synthetic"] = proba
df["pred"] = (proba >= 0.5).astype(int)
df["correct"] = df["pred"] == df["is_llm"]
df["confidence"] = np.where(df["is_llm"] == 1, df["p_synthetic"], 1 - df["p_synthetic"])

print("Accuracy:", round(df["correct"].mean(), 3))
print("Errors:", (~df["correct"]).sum())

## Step 3: Where the errors occur
Error rates are broken down by source and by group, which shows whether one generator is markedly harder to recognise than the other.

In [ ]:
err = df.groupby("source")["correct"].agg(n="count", correct="sum")
err["errors"] = err["n"] - err["correct"]
err["error_rate"] = (err["errors"] / err["n"]).round(3)
print(err[["n", "errors", "error_rate"]])
print()
print(pd.crosstab(df["source"], df["correct"], normalize="index").round(3))

## Step 4: Feature profile of the misclassified texts
Misclassified texts are compared with the medians of both classes. A human review taken for synthetic is expected to resemble the synthetic profile on the features that carry the most weight.

In [ ]:
def profile(mask, name):
    return df[mask][FEATURES].median().rename(name)

prof = pd.concat([
    profile(df.source == "human", "Human (all)"),
    profile((df.source == "human") & (~df.correct), "Human misclassified"),
    profile(df.is_llm == 1, "Synthetic (all)"),
    profile((df.is_llm == 1) & (~df.correct), "Synthetic misclassified"),
], axis=1).round(3)
prof.to_csv("results/e5_error_profile.csv")
print(prof)

## Step 5: The most confident errors
The texts below are those the classifier got wrong while being most certain. They are the material for the qualitative discussion on the poster.

In [ ]:
errors = df[~df["correct"]].sort_values("confidence").copy()
errors["true"] = np.where(errors["is_llm"] == 1, "Synthetic", "Human")
errors["predicted"] = np.where(errors["pred"] == 1, "Synthetic", "Human")

show = ["text_id", "source", "primaryTitle", "true", "predicted", "p_synthetic",
        "pron_ratio", "quest_rate", "noun_ratio", "ttr"]
print(errors[show].round(3).to_string(index=False))

In [ ]:
for _, r in errors.head(6).iterrows():
    print("=" * 90)
    print(f"{r.text_id}  |  {r.primaryTitle}  |  true={r.true}  predicted={r.predicted}  "
          f"p(synthetic)={r.p_synthetic:.2f}")
    print(f"pronoun={r.pron_ratio:.3f}  noun={r.noun_ratio:.3f}  ttr={r.ttr:.3f}  "
          f"question={r.quest_rate:.2f}")
    print()
    print(r.text[:600])
    print()

## Step 6: Export the table for the poster
Five examples are exported with a short excerpt each. The interpretation column is filled in by hand after reading the texts above — that reading is the analysis, and it should not be automated.

In [ ]:
table = errors.head(5)[["text_id", "primaryTitle", "true", "predicted", "p_synthetic"]].copy()
table["excerpt"] = errors.head(5)["text"].str.slice(0, 160) + "..."
table["interpretation"] = ""          # 手动填写
table.round(2).to_csv("results/e5_error_examples.csv", index=False)
print(table.to_string(index=False))

In [ ]:
files.download("results/e5_error_examples.csv")
files.download("results/e5_error_profile.csv")